# Offline Trace MOPSO - DI + ART Routing Simulation


## Depedensi


In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "matplotlib", "scikit-fuzzy", "tqdm"],
    check=True,
)

if os.environ.get("COLAB_GPU"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=False)


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

os.environ.setdefault("MPLCONFIGDIR", "/tmp")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import cupy as cp
    _GPU_COUNT = int(cp.cuda.runtime.getDeviceCount())
except Exception:
    cp = None
    _GPU_COUNT = 0

GPU_AVAILABLE = (cp is not None) and (_GPU_COUNT > 0)
GPU_ENABLED = GPU_AVAILABLE


def set_training_backend(prefer_gpu: bool = True) -> str:
    global GPU_ENABLED
    GPU_ENABLED = bool(prefer_gpu and GPU_AVAILABLE)
    return "gpu" if GPU_ENABLED else "cpu"


def current_training_backend() -> str:
    return "gpu" if GPU_ENABLED else "cpu"


def get_xp(use_gpu: Optional[bool] = None):
    enable_gpu = GPU_ENABLED if use_gpu is None else bool(use_gpu and GPU_AVAILABLE)
    if enable_gpu and cp is not None:
        return cp
    return np


def to_numpy(value):
    if cp is not None and isinstance(value, cp.ndarray):
        return cp.asnumpy(value)
    return np.asarray(value)

try:
    import skfuzzy as fuzz
except Exception:
    class _FallbackFuzz(object):
        @staticmethod
        def trimf(x, abc):
            x = np.asarray(x, dtype=np.float64)
            a, b, c = abc
            y = np.zeros_like(x, dtype=np.float64)
            if b > a:
                left = (x > a) & (x < b)
                y[left] = (x[left] - a) / (b - a)
            if c > b:
                right = (x > b) & (x < c)
                y[right] = np.maximum(y[right], (c - x[right]) / (c - b))
            y[x == b] = 1.0
            return np.clip(y, 0.0, 1.0)

    fuzz = _FallbackFuzz()

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        if iterable is None:
            class _NullTqdm(object):
                def update(self, _n=1):
                    return None

                def set_postfix(self, **_kwargs):
                    return None

                def close(self):
                    return None

            return _NullTqdm()
        return iterable



## Konstanta


In [ ]:
# ----------------------------

DIMENSIONS = 27
NUM_PARTICLES_DEFAULT = 100
ITERATIONS_DEFAULT = 500
MAX_ARCHIVE = 128
INERTIA_MAX_W = 0.90
INERTIA_MIN_W = 0.40
COGNITIVE_C1 = 1.5
SOCIAL_C2 = 1.5

# Rule base terkompilasi (identik dengan compiledRules di Go)
# tuple: (cpu_idx, queue_idx, resp_idx, out_idx)
COMPILED_RULES: Tuple[Tuple[int, int, int, int], ...] = (
    (0, 0, 0, 2), (0, 0, 1, 2), (0, 0, 2, 1), (0, 1, 0, 2), (0, 1, 1, 1), (0, 1, 2, 1), (0, 2, 0, 1), (0, 2, 1, 1), (0, 2, 2, 0),
    (1, 0, 0, 2), (1, 0, 1, 1), (1, 0, 2, 1), (1, 1, 0, 1), (1, 1, 1, 1), (1, 1, 2, 0), (1, 2, 0, 1), (1, 2, 1, 0), (1, 2, 2, 0),
    (2, 0, 0, 1), (2, 0, 1, 1), (2, 0, 2, 0), (2, 1, 0, 1), (2, 1, 1, 0), (2, 1, 2, 0), (2, 2, 0, 0), (2, 2, 1, 0), (2, 2, 2, 0),
)

# Output MF triangles asimetris: index 0=Rendah,1=Sedang,2=Tinggi
OUT_MF = np.array([[0.0, 0.0, 50.0], [25.0, 50.0, 75.0], [50.0, 100.0, 100.0]], dtype=np.float64)

DEFAULT_BASE_PARAMS = np.array(
    [
        0.0, 0.0, 50.0, 0.0, 50.0, 100.0, 50.0, 100.0, 100.0,
        0.0, 0.0, 500.0, 0.0, 500.0, 1000.0, 500.0, 1000.0, 1000.0,
        0.0, 0.0, 500.0, 0.0, 500.0, 1000.0, 500.0, 1000.0, 1000.0
    ],
    dtype=np.float64,
)
LOWER_BOUNDS = np.zeros(DIMENSIONS, dtype=np.float64)
UPPER_BOUNDS = np.array([100.0] * 9 + [1000.0] * 18, dtype=np.float64)
GPU_TRAINING_CHUNK_ROWS = 2048
CPU_TRAINING_CHUNK_ROWS = 8192
DEFAULT_NODE1_CPU_CAPACITY = 100.0
DEFAULT_NODE2_CPU_CAPACITY = 50.0
DI_ART_TIEBREAK_WEIGHT = 1e-6
STAGNATION_PATIENCE = 75
RESEED_FRACTION = 0.20
RESEED_SCALE_MULTIPLIER = 2.0

print("Konstanta training siap:")
print(pd.DataFrame([
    {"item": "DIMENSIONS", "value": DIMENSIONS},
    {"item": "INERTIA_MAX_W", "value": INERTIA_MAX_W},
    {"item": "INERTIA_MIN_W", "value": INERTIA_MIN_W},
    {"item": "UPPER_BOUNDS_CPU_MAX", "value": float(np.max(UPPER_BOUNDS[:9]))},
    {"item": "UPPER_BOUNDS_QUEUE_RT_MAX", "value": float(np.max(UPPER_BOUNDS[9:]))},
    {"item": "CPU_TRAINING_CHUNK_ROWS", "value": CPU_TRAINING_CHUNK_ROWS},
    {"item": "GPU_TRAINING_CHUNK_ROWS", "value": GPU_TRAINING_CHUNK_ROWS},
    {"item": "DEFAULT_BACKEND", "value": current_training_backend()},
    {"item": "GPU_AVAILABLE", "value": GPU_AVAILABLE},
]).to_string(index=False))



## Data model


In [ ]:
# ----------------------------

@dataclass
class OfflineDataset:
    scenario_name: str
    timestamp: np.ndarray
    total_requests: np.ndarray
    node1_requests: np.ndarray
    node2_requests: np.ndarray
    node1_cpu_raw: np.ndarray
    node2_cpu_raw: np.ndarray
    node1_cpu_norm: np.ndarray
    node2_cpu_norm: np.ndarray
    node1_cpu_capacity: np.ndarray
    node2_cpu_capacity: np.ndarray
    node1_queue: np.ndarray
    node2_queue: np.ndarray
    node1_response_ms: np.ndarray
    node2_response_ms: np.ndarray

    @property
    def sample_count(self) -> int:
        return int(self.total_requests.shape[0])

    @property
    def usable_mask(self) -> np.ndarray:
        return self.total_requests > 0

    @property
    def used_samples(self) -> int:
        return int(np.count_nonzero(self.usable_mask))


@dataclass
class SystemConstants:
    node1_cpu_base: float
    node1_cpu_cost: float
    node2_cpu_base: float
    node2_cpu_cost: float
    node1_rt_base: float
    node1_rt_cost: float
    node2_rt_base: float
    node2_rt_cost: float


@dataclass
class OfflineObjective:
    di: float
    art: float


@dataclass
class ArchiveScoreRow:
    index: int
    params: List[float]
    di: float
    art: float
    norm_di: float
    norm_art: float
    score: float


@dataclass
class OfflineSolution:
    params: List[float]
    objective: OfflineObjective


@dataclass
class OfflineConfig:
    particles: int = NUM_PARTICLES_DEFAULT
    iterations: int = ITERATIONS_DEFAULT
    initial_spread: float = 8.0
    seed: int = 0

    def normalized(self) -> "OfflineConfig":
        particles = self.particles if self.particles > 0 else NUM_PARTICLES_DEFAULT
        iterations = self.iterations if self.iterations > 0 else ITERATIONS_DEFAULT
        spread = self.initial_spread if self.initial_spread > 0 else 8.0
        seed = self.seed if self.seed != 0 else int(time.time_ns())
        return OfflineConfig(particles=particles, iterations=iterations, initial_spread=spread, seed=seed)


@dataclass
class OfflineResult:
    generated_at: str
    sample_count: int
    used_samples: int
    config: OfflineConfig
    archive: List[OfflineSolution]
    best_balanced: OfflineSolution
    best_di: OfflineSolution
    best_art: OfflineSolution
    best_balanced_score: float
    system_constants: SystemConstants
    archive_trace: List[ArchiveScoreRow] = field(default_factory=list)
    normalization_stats: Dict[str, float] = field(default_factory=dict)
    score_history: List[float] = field(default_factory=list)
    iter_best_history: List[float] = field(default_factory=list)
    iter_mean_history: List[float] = field(default_factory=list)
    archive_size_history: List[float] = field(default_factory=list)
    stagnation_events: int = 0


print("Data model siap:")
print(pd.DataFrame([
    {"model": "OfflineDataset", "fields": len(OfflineDataset.__annotations__), "focus": "ground truth per snapshot"},
    {"model": "SystemConstants", "fields": len(SystemConstants.__annotations__), "focus": "regression constants"},
    {"model": "OfflineObjective", "fields": len(OfflineObjective.__annotations__), "focus": "DI + ART objective"},
    {"model": "ArchiveScoreRow", "fields": len(ArchiveScoreRow.__annotations__), "focus": "pareto archive ranking"},
    {"model": "OfflineResult", "fields": len(OfflineResult.__annotations__), "focus": "training summary + plots"},
]).to_string(index=False))



## Helper util


In [ ]:
# ----------------------------

def clamp(v: np.ndarray | float, lo: float, hi: float):
    return np.clip(v, lo, hi)


def lower_bound(_d: int) -> float:
    return 0.0


def upper_bound(d: int) -> float:
    if d <= 8:
        return 100.0
    if d <= 17:
        return 1000.0
    return 1000.0


def max_int(a: int, b: int) -> int:
    return a if a > b else b


def max_int64_array(x: np.ndarray, minimum: int = 1) -> np.ndarray:
    return np.maximum(x, minimum)


def to_raw_cpu(cpu: np.ndarray, capacity: np.ndarray) -> np.ndarray:
    cap = np.where(capacity <= 0, 100.0, capacity)
    return np.clip(cpu, 0.0, cap)


def to_normalized_cpu(cpu: np.ndarray, capacity: np.ndarray) -> np.ndarray:
    cap = np.where(capacity <= 0, 100.0, capacity)
    raw = to_raw_cpu(cpu, cap)
    norm = (raw / cap) * 100.0
    return np.clip(norm, 0.0, 100.0)


def di_ratio(cpu1: np.ndarray, cpu2: np.ndarray) -> np.ndarray:
    diff = np.abs(cpu1 - cpu2) / 100.0
    return np.clip(diff, 0.0, 1.0)


def _slugify_plot_title(title: str) -> str:
    text = str(title).strip().lower()
    safe = "".join(ch if ch.isalnum() else "_" for ch in text)
    safe = "_".join(part for part in safe.split("_") if part)
    return safe or "plot"


def plot_fuzzy_membership(params: np.ndarray, title: str, out_dir: str | Path) -> Path:
    p = repair_params(np.asarray(params, dtype=np.float64))
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"membership_{_slugify_plot_title(title)}.png"

    groups = [
        ("Input CPU", p[0:9], 0.0, 100.0, "Normalized CPU (%)"),
        ("Input Queue", p[9:18], 0.0, 1000.0, "Queue Length"),
        ("Input Response Time", p[18:27], 0.0, 1000.0, "Response Time (ms)"),
    ]
    labels = ["Low", "Medium", "High"]
    colors = ["#1f77b4", "#ff7f0e", "#d62728"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
    for ax, (panel_title, values, x_lo, x_hi, xlabel) in zip(axes, groups):
        x = np.linspace(x_lo, x_hi, 600)
        triplets = values.reshape(3, 3)
        for mf_idx, (label, color) in enumerate(zip(labels, colors)):
            if mf_idx == 0:
                y = fuzzify_left(x, triplets[mf_idx][0], triplets[mf_idx][1], triplets[mf_idx][2], xp_module=np)
            elif mf_idx == 1:
                y = fuzzify_triangle(x, triplets[mf_idx][0], triplets[mf_idx][1], triplets[mf_idx][2], xp_module=np)
            else:
                y = fuzzify_right(x, triplets[mf_idx][0], triplets[mf_idx][1], triplets[mf_idx][2], xp_module=np)
            ax.plot(x, y, linewidth=2.2, color=color, label=label)
            ax.fill_between(x, 0.0, y, color=color, alpha=0.10)
        ax.set_title(panel_title)
        ax.set_xlabel(xlabel)
        ax.set_ylim(0.0, 1.05)
        ax.grid(True, linestyle="--", alpha=0.25)
    axes[0].set_ylabel("Membership Degree")
    axes[-1].legend(loc="upper right")
    fig.suptitle(title)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def plot_snapshot_trace(
    params: np.ndarray,
    ds: OfflineDataset,
    constants: SystemConstants,
    title: str,
    out_dir: str | Path,
) -> Path:
    sim = _simulate_routing_response(params, ds, constants)
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"snapshot_trace_{_slugify_plot_title(title)}.png"

    x = np.arange(ds.sample_count)
    active = ds.total_requests > 0

    def active_only(values: np.ndarray) -> np.ndarray:
        arr = np.asarray(values, dtype=np.float64)
        return np.where(active, arr, np.nan)

    fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
    node1_color = "#1f77b4"
    node2_color = "#ff7f0e"
    di_color = "#2ca02c"

    axes[0].plot(x, active_only(sim["pred_cpu1"]), color=node1_color, linewidth=1.8, label="Node 1")
    axes[0].plot(x, active_only(sim["pred_cpu2"]), color=node2_color, linewidth=1.8, label="Node 2")
    axes[0].set_ylabel("CPU Norm (%)")
    axes[0].set_title("Predicted CPU per Snapshot")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, linestyle="--", alpha=0.25)

    axes[1].plot(x, active_only(sim["pred_rt1"]), color=node1_color, linewidth=1.8, label="Node 1")
    axes[1].plot(x, active_only(sim["pred_rt2"]), color=node2_color, linewidth=1.8, label="Node 2")
    axes[1].set_ylabel("Response Time (ms)")
    axes[1].set_title("Predicted Response Time per Snapshot")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, linestyle="--", alpha=0.25)

    axes[2].plot(x, active_only(sim["pred_req1"]), color=node1_color, linewidth=1.8, label="Node 1 Routed Req")
    axes[2].plot(x, active_only(sim["pred_req2"]), color=node2_color, linewidth=1.8, label="Node 2 Routed Req")
    axes[2].set_ylabel("Routed Requests")
    axes[2].set_title("Predicted Routed Requests per Snapshot")
    axes[2].legend(loc="upper right")
    axes[2].grid(True, linestyle="--", alpha=0.25)

    axes[3].plot(x, active_only(sim["di_per_row"]), color=di_color, linewidth=2.0, label="DI Ratio")
    axes[3].set_ylabel("DI Ratio")
    axes[3].set_xlabel("Snapshot Index")
    axes[3].set_ylim(0.0, 1.05)
    axes[3].set_title("Capacity-Normalized DI per Snapshot")
    axes[3].legend(loc="upper right")
    axes[3].grid(True, linestyle="--", alpha=0.25)

    fig.suptitle(title)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def objective_score(obj: OfflineObjective) -> float:
    # DI ratio adalah objective utama; ART hanya tie-break kecil agar selection tetap konsisten.
    return obj.di + ((obj.art / 1000.0) * DI_ART_TIEBREAK_WEIGHT)


def dominates(a: OfflineObjective, b: OfflineObjective) -> bool:
    # Pareto murni: minimisasi DI dan ART.
    better_or_equal = (a.di <= b.di) and (a.art <= b.art)
    strictly_better = (a.di < b.di) or (a.art < b.art)
    return bool(better_or_equal and strictly_better)


def minmax_scale(values: np.ndarray) -> Tuple[np.ndarray, float, float]:
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return values, 0.0, 0.0
    min_v = float(np.min(values))
    max_v = float(np.max(values))
    den = max(max_v - min_v, 1e-9)
    scaled = (values - min_v) / den
    return scaled, min_v, max_v


def minmax_scale_value(value: float, min_v: float, max_v: float) -> float:
    scaled = (float(value) - float(min_v)) / max(max_v - min_v, 1e-9)
    return float(np.clip(scaled, 0.0, 1.0))


def balanced_score_from_normalized(norm_di: float, norm_art: float) -> float:
    # Ranking arsip DI-first: ART hanya membantu memilih kandidat dengan DI hampir sama.
    return norm_di + (DI_ART_TIEBREAK_WEIGHT * norm_art)


def archive_trace_rows(archive: Sequence[OfflineSolution]) -> tuple[list[ArchiveScoreRow], Dict[str, float]]:
    if len(archive) == 0:
        return [], {}

    di_values = np.array([sol.objective.di for sol in archive], dtype=np.float64)
    art_values = np.array([sol.objective.art for sol in archive], dtype=np.float64)
    norm_di, min_di, max_di = minmax_scale(di_values)
    norm_art, min_art, max_art = minmax_scale(art_values)
    scores = balanced_score_from_normalized(norm_di, norm_art)

    rows = [
        ArchiveScoreRow(
            index=i,
            params=clone_params(np.asarray(sol.params, dtype=np.float64)).tolist(),
            di=float(sol.objective.di),
            art=float(sol.objective.art),
            norm_di=float(norm_di[i]),
            norm_art=float(norm_art[i]),
            score=float(scores[i]),
        )
        for i, sol in enumerate(archive)
    ]
    stats = {
        "min_di": min_di,
        "max_di": max_di,
        "min_art": min_art,
        "max_art": max_art,
    }
    return rows, stats


def select_best_balanced_solution(archive: Sequence[OfflineSolution]) -> tuple[OfflineSolution, float, list[ArchiveScoreRow], Dict[str, float]]:
    rows, stats = archive_trace_rows(archive)
    if len(rows) == 0:
        raise ValueError("pareto archive kosong")
    best_row = min(rows, key=lambda row: (row.di, row.art, row.score))
    best = archive[best_row.index]
    stats["best_index"] = float(best_row.index)
    stats["best_score"] = float(best_row.score)
    return best, float(best_row.score), rows, stats


def find_trace_row(rows: Sequence[ArchiveScoreRow], params: Sequence[float]) -> Optional[ArchiveScoreRow]:
    target = np.asarray(params, dtype=np.float64)
    for row in rows:
        if np.allclose(np.asarray(row.params, dtype=np.float64), target, atol=1e-9, rtol=1e-9):
            return row
    return None


def clone_params(params: np.ndarray) -> np.ndarray:
    out = np.asarray(params, dtype=np.float64).copy()
    if out.shape[0] != DIMENSIONS:
        raise ValueError(f"Panjang parameter harus {DIMENSIONS}, dapat {out.shape[0]}")
    return out


## Param


In [ ]:
# ----------------------------

def domain_shape_constraints(start: int) -> tuple[float, float, float, float]:
    lo = lower_bound(start)
    hi = upper_bound(start)
    if start <= 8:
        return lo, hi, 15.0, 25.0
    return lo, hi, 150.0, 200.0


def _project_ordered_peaks(low_peak: float, med_peak: float, high_peak: float, lo: float, hi: float, peak_gap: float) -> tuple[float, float, float]:
    low_peak = float(np.clip(low_peak, lo, hi - (2.0 * peak_gap)))
    med_peak = float(np.clip(med_peak, lo + peak_gap, hi - peak_gap))
    high_peak = float(np.clip(high_peak, lo + (2.0 * peak_gap), hi))

    med_peak = max(med_peak, low_peak + peak_gap)
    high_peak = max(high_peak, med_peak + peak_gap)

    if high_peak > hi:
        high_peak = hi
    med_peak = min(med_peak, high_peak - peak_gap)
    low_peak = min(low_peak, med_peak - peak_gap)

    if low_peak < lo:
        low_peak = lo
    med_peak = max(med_peak, low_peak + peak_gap)
    high_peak = max(high_peak, med_peak + peak_gap)

    if high_peak > hi:
        high_peak = hi
        med_peak = min(med_peak, high_peak - peak_gap)
        low_peak = min(low_peak, med_peak - peak_gap)

    return float(low_peak), float(med_peak), float(high_peak)


def _repair_domain(params: np.ndarray, start: int) -> None:
    lo, hi, slope_min, peak_gap = domain_shape_constraints(start)

    low_peak, med_peak, high_peak = _project_ordered_peaks(
        float(params[start + 1]),
        float(params[start + 4]),
        float(params[start + 7]),
        lo,
        hi,
        peak_gap,
    )

    low_zero = float(np.clip(params[start + 2], low_peak + slope_min, med_peak))
    med_left_max = min(low_zero, med_peak - slope_min)
    med_left = float(np.clip(params[start + 3], low_peak, med_left_max))
    high_left = float(np.clip(params[start + 6], med_peak, high_peak - slope_min))
    med_right_min = max(high_left, med_peak + slope_min)
    med_right = float(np.clip(params[start + 5], med_right_min, high_peak))

    params[start] = lo
    params[start + 1] = low_peak
    params[start + 2] = low_zero
    params[start + 3] = med_left
    params[start + 4] = med_peak
    params[start + 5] = med_right
    params[start + 6] = high_left
    params[start + 7] = high_peak
    params[start + 8] = hi


def repair_params(params: np.ndarray) -> np.ndarray:
    p = clone_params(params)
    for start in (0, 9, 18):
        _repair_domain(p, start)
    return np.round(p, 6)


def _is_sane_domain(params: np.ndarray, start: int, tol: float = 1e-6) -> bool:
    lo, hi, slope_min, peak_gap = domain_shape_constraints(start)

    low_a = float(params[start])
    low_peak = float(params[start + 1])
    low_zero = float(params[start + 2])
    med_left = float(params[start + 3])
    med_peak = float(params[start + 4])
    med_right = float(params[start + 5])
    high_left = float(params[start + 6])
    high_peak = float(params[start + 7])
    high_c = float(params[start + 8])

    if abs(low_a - lo) > tol or abs(high_c - hi) > tol:
        return False
    if low_peak < lo - tol or high_peak > hi + tol:
        return False

    if not ((low_peak + tol) < low_zero):
        return False
    if not ((med_left + tol) < med_peak and (med_peak + tol) < med_right):
        return False
    if not ((high_left + tol) < high_peak):
        return False

    if ((low_zero - low_peak) + tol) < slope_min:
        return False
    if ((med_peak - med_left) + tol) < slope_min or ((med_right - med_peak) + tol) < slope_min:
        return False
    if ((high_peak - high_left) + tol) < slope_min:
        return False

    if ((med_peak - low_peak) + tol) < peak_gap or ((high_peak - med_peak) + tol) < peak_gap:
        return False

    if low_zero > med_peak + tol:
        return False
    if med_left < low_peak - tol or med_left > low_zero + tol:
        return False
    if high_left < med_peak - tol or high_left > med_right + tol:
        return False
    if med_right > high_peak + tol:
        return False

    return True


def is_sane_params(params: np.ndarray) -> bool:
    if params.shape[0] != DIMENSIONS:
        return False
    return all(_is_sane_domain(params, start) for start in (0, 9, 18))


## Fuzzy engine (vectorized)


In [ ]:
# ----------------------------

try:
    cp
except NameError:
    cp = None


def _resolve_xp(*values, xp_module=None):
    if xp_module is not None:
        return xp_module
    if cp is not None:
        for value in values:
            if isinstance(value, cp.ndarray):
                return cp
    return np


def fuzzify_left(v: np.ndarray, _a: float, b: float, c: float, xp_module=None) -> np.ndarray:
    xp = _resolve_xp(v, b, c, xp_module=xp_module)
    v = xp.asarray(v, dtype=xp.float64)
    b = xp.asarray(b, dtype=xp.float64)
    c = xp.asarray(c, dtype=xp.float64)

    out = xp.ones_like(v, dtype=xp.float64)
    out = xp.where(v >= c, 0.0, out)
    den = c - b
    safe_den = xp.where(den > 0.0, den, 1.0)
    ramp = (c - v) / safe_den
    mid = (v > b) & (v < c) & (den > 0.0)
    out = xp.where(mid, ramp, out)
    return xp.where(den <= 0.0, xp.where(v <= b, 1.0, 0.0), out)


def fuzzify_triangle(v: np.ndarray, a: float, b: float, c: float, xp_module=None) -> np.ndarray:
    xp = _resolve_xp(v, a, b, c, xp_module=xp_module)
    v = xp.asarray(v, dtype=xp.float64)
    a = xp.asarray(a, dtype=xp.float64)
    b = xp.asarray(b, dtype=xp.float64)
    c = xp.asarray(c, dtype=xp.float64)

    left_den = xp.where(b > a, b - a, 1.0)
    right_den = xp.where(c > b, c - b, 1.0)
    left = xp.where((v > a) & (v < b), (v - a) / left_den, 0.0)
    right = xp.where((v > b) & (v < c), (c - v) / right_den, 0.0)
    out = xp.maximum(left, right)
    out = xp.where(v == b, 1.0, out)
    out = xp.where((b <= a) & (v <= b), 1.0, out)
    out = xp.where((c <= b) & (v >= b), 1.0, out)
    return xp.clip(out, 0.0, 1.0)


def fuzzify_right(v: np.ndarray, a: float, b: float, _c: float, xp_module=None) -> np.ndarray:
    xp = _resolve_xp(v, a, b, xp_module=xp_module)
    v = xp.asarray(v, dtype=xp.float64)
    a = xp.asarray(a, dtype=xp.float64)
    b = xp.asarray(b, dtype=xp.float64)

    out = xp.ones_like(v, dtype=xp.float64)
    out = xp.where(v <= a, 0.0, out)
    den = b - a
    safe_den = xp.where(den > 0.0, den, 1.0)
    ramp = (v - a) / safe_den
    mid = (v > a) & (v < b) & (den > 0.0)
    out = xp.where(mid, ramp, out)
    return xp.where(den <= 0.0, xp.where(v >= b, 1.0, 0.0), out)


def _get_fuzzy_engine_cache(xp_module):
    backend_key = "gpu" if (cp is not None and xp_module is cp) else "cpu"
    cache = getattr(_get_fuzzy_engine_cache, "_cache", {})
    if backend_key not in cache:
        rules = xp_module.asarray(COMPILED_RULES, dtype=xp_module.intp)
        z = xp_module.arange(0.0, 101.0, 1.0, dtype=xp_module.float64)
        base_mf = xp_module.stack(
            tuple(
                fuzzify_triangle(
                    z,
                    float(a),
                    float(b),
                    float(c),
                    xp_module=xp_module,
                )
                for a, b, c in OUT_MF
            ),
            axis=0,
        ).astype(xp_module.float64)
        rule_groups = tuple(
            xp_module.asarray(
                np.flatnonzero(np.asarray([rule[3] for rule in COMPILED_RULES], dtype=np.intp) == out_idx),
                dtype=xp_module.intp,
            )
            for out_idx in range(3)
        )
        cache[backend_key] = (rules, z, base_mf, rule_groups)
        _get_fuzzy_engine_cache._cache = cache
    return cache[backend_key]


def fuzzy_score_vectorized_batch(params: np.ndarray, cpu: np.ndarray, q: np.ndarray, rt: np.ndarray, xp_module=None) -> np.ndarray:
    xp = _resolve_xp(params, cpu, q, rt, xp_module=xp_module)
    p = xp.asarray(params, dtype=xp.float64)
    single = p.ndim == 1
    if single:
        p = p[None, :]

    cpu = xp.asarray(cpu, dtype=xp.float64)
    q = xp.asarray(q, dtype=xp.float64)
    rt = xp.asarray(rt, dtype=xp.float64)
    rules, z, base_mf, rule_groups = _get_fuzzy_engine_cache(xp)

    cpu_view = cpu[None, :]
    q_view = q[None, :]
    rt_view = rt[None, :]

    mu_cpu = xp.stack(
        (
            fuzzify_left(cpu_view, p[:, 0, None], p[:, 1, None], p[:, 2, None], xp_module=xp),
            fuzzify_triangle(cpu_view, p[:, 3, None], p[:, 4, None], p[:, 5, None], xp_module=xp),
            fuzzify_right(cpu_view, p[:, 6, None], p[:, 7, None], p[:, 8, None], xp_module=xp),
        ),
        axis=1,
    )
    mu_q = xp.stack(
        (
            fuzzify_left(q_view, p[:, 9, None], p[:, 10, None], p[:, 11, None], xp_module=xp),
            fuzzify_triangle(q_view, p[:, 12, None], p[:, 13, None], p[:, 14, None], xp_module=xp),
            fuzzify_right(q_view, p[:, 15, None], p[:, 16, None], p[:, 17, None], xp_module=xp),
        ),
        axis=1,
    )
    mu_r = xp.stack(
        (
            fuzzify_left(rt_view, p[:, 18, None], p[:, 19, None], p[:, 20, None], xp_module=xp),
            fuzzify_triangle(rt_view, p[:, 21, None], p[:, 22, None], p[:, 23, None], xp_module=xp),
            fuzzify_right(rt_view, p[:, 24, None], p[:, 25, None], p[:, 26, None], xp_module=xp),
        ),
        axis=1,
    )

    rule_strengths = xp.minimum(
        xp.minimum(
            xp.take(mu_cpu, rules[:, 0], axis=1),
            xp.take(mu_q, rules[:, 1], axis=1),
        ),
        xp.take(mu_r, rules[:, 2], axis=1),
    )

    alpha_out = xp.stack(
        tuple(xp.max(xp.take(rule_strengths, group, axis=1), axis=1) for group in rule_groups),
        axis=1,
    )

    union_area = xp.zeros((p.shape[0], cpu.shape[0], z.shape[0]), dtype=xp.float64)
    for out_idx in range(3):
        union_area = xp.fmax(
            union_area,
            xp.fmin(alpha_out[:, out_idx, :, None], base_mf[out_idx][None, None, :]),
        )

    numerator = xp.sum(z[None, None, :] * union_area, axis=-1)
    denominator = xp.sum(union_area, axis=-1)
    valid_mask = denominator > 0.0
    safe_denominator = xp.where(valid_mask, denominator, 1.0)
    scores = xp.where(valid_mask, numerator / safe_denominator, 0.0)
    return scores[0] if single else scores


def fuzzy_score_vectorized(params: np.ndarray, cpu: np.ndarray, q: np.ndarray, rt: np.ndarray) -> np.ndarray:
    return fuzzy_score_vectorized_batch(params, cpu, q, rt, xp_module=np)


## Fitness evaluator (DI + ART routing simulation)


In [ ]:
# ----------------------------

def _fit_linear_model(requests: np.ndarray, observed: np.ndarray, fallback: float = 0.0) -> tuple[float, float]:
    x = _sanitize_float_array(np.asarray(requests, dtype=np.float64), 0.0)
    y = _sanitize_float_array(np.asarray(observed, dtype=np.float64), fallback)
    mask = np.isfinite(x) & np.isfinite(y) & (x >= 0.0)

    if not np.any(mask):
        return float(max(fallback, 0.0)), 0.0

    x = x[mask]
    y = y[mask]

    if x.size < 2 or float(np.ptp(x)) <= 1e-9:
        base = float(np.median(y))
        cost = 0.0
    else:
        design = np.column_stack([np.ones_like(x), x])
        coeffs, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
        base = float(coeffs[0])
        cost = float(coeffs[1])

    if not np.isfinite(base):
        base = float(np.median(y)) if y.size > 0 else float(fallback)
    if not np.isfinite(cost) or cost < 0.0:
        cost = 0.0
    if base < 0.0:
        base = 0.0

    return float(base), float(cost)


def extract_system_constants(ds: OfflineDataset) -> SystemConstants:
    node1_cpu_base, node1_cpu_cost = _fit_linear_model(ds.node1_requests, ds.node1_cpu_raw, 0.0)
    node2_cpu_base, node2_cpu_cost = _fit_linear_model(ds.node2_requests, ds.node2_cpu_raw, 0.0)
    node1_rt_base, node1_rt_cost = _fit_linear_model(ds.node1_requests, ds.node1_response_ms, 1.0)
    node2_rt_base, node2_rt_cost = _fit_linear_model(ds.node2_requests, ds.node2_response_ms, 1.0)

    return SystemConstants(
        node1_cpu_base=float(node1_cpu_base),
        node1_cpu_cost=float(node1_cpu_cost),
        node2_cpu_base=float(node2_cpu_base),
        node2_cpu_cost=float(node2_cpu_cost),
        node1_rt_base=float(node1_rt_base),
        node1_rt_cost=float(node1_rt_cost),
        node2_rt_base=float(node2_rt_base),
        node2_rt_cost=float(node2_rt_cost),
    )


def _simulate_routing_response(
    params: np.ndarray,
    ds: OfflineDataset,
    constants: Optional[SystemConstants] = None,
) -> Dict[str, np.ndarray | SystemConstants]:
    p = repair_params(params)
    constants = extract_system_constants(ds) if constants is None else constants

    node1_score = fuzzy_score_vectorized(p, ds.node1_cpu_norm, ds.node1_queue, ds.node1_response_ms)
    node2_score = fuzzy_score_vectorized(p, ds.node2_cpu_norm, ds.node2_queue, ds.node2_response_ms)

    score_sum = node1_score + node2_score
    safe_sum = np.where(score_sum > 1e-9, score_sum, 1.0)
    weight1 = np.where(score_sum > 1e-9, node1_score / safe_sum, 0.5)
    weight2 = np.where(score_sum > 1e-9, node2_score / safe_sum, 0.5)

    pred_req1 = weight1 * ds.total_requests
    pred_req2 = weight2 * ds.total_requests

    pred_cpu1_raw = np.clip(constants.node1_cpu_base + (pred_req1 * constants.node1_cpu_cost), 0.0, ds.node1_cpu_capacity)
    pred_cpu2_raw = np.clip(constants.node2_cpu_base + (pred_req2 * constants.node2_cpu_cost), 0.0, ds.node2_cpu_capacity)
    pred_cpu1 = to_normalized_cpu(pred_cpu1_raw, ds.node1_cpu_capacity)
    pred_cpu2 = to_normalized_cpu(pred_cpu2_raw, ds.node2_cpu_capacity)
    pred_rt1 = np.maximum(constants.node1_rt_base + (pred_req1 * constants.node1_rt_cost), 0.0)
    pred_rt2 = np.maximum(constants.node2_rt_base + (pred_req2 * constants.node2_rt_cost), 0.0)

    di_per_row = di_ratio(pred_cpu1, pred_cpu2)
    art_per_row = np.where(
        ds.total_requests > 0,
        ((pred_rt1 * pred_req1) + (pred_rt2 * pred_req2)) / np.maximum(ds.total_requests, 1e-9),
        0.0,
    )
    mean_rt_per_row = (pred_rt1 + pred_rt2) / 2.0

    return {
        "params": p,
        "constants": constants,
        "node1_score": node1_score,
        "node2_score": node2_score,
        "weight1": weight1,
        "weight2": weight2,
        "pred_req1": pred_req1,
        "pred_req2": pred_req2,
        "pred_cpu1_raw": pred_cpu1_raw,
        "pred_cpu2_raw": pred_cpu2_raw,
        "pred_cpu1": pred_cpu1,
        "pred_cpu2": pred_cpu2,
        "pred_rt1": pred_rt1,
        "pred_rt2": pred_rt2,
        "di_per_row": di_per_row,
        "art_per_row": art_per_row,
        "mean_rt_per_row": mean_rt_per_row,
    }


def evaluate_dataset_di_art(
    params: np.ndarray,
    ds: OfflineDataset,
    constants: Optional[SystemConstants] = None,
) -> OfflineObjective:
    active = ds.total_requests > 0
    if not np.any(active):
        return OfflineObjective(di=1.0, art=0.0)

    sim = _simulate_routing_response(params, ds, constants)
    di_value = float(np.mean(sim["di_per_row"][active]))
    art_value = float(np.mean(sim["art_per_row"][active]))
    return OfflineObjective(di=di_value, art=art_value)


def evaluate_historical_base_objective(ds: OfflineDataset) -> OfflineObjective:
    active = ds.total_requests > 0
    if not np.any(active):
        return OfflineObjective(di=0.0, art=0.0)

    di_per_row = di_ratio(ds.node1_cpu_norm, ds.node2_cpu_norm)
    art_per_row = np.where(
        ds.total_requests > 0,
        ((ds.node1_response_ms * ds.node1_requests) + (ds.node2_response_ms * ds.node2_requests)) / np.maximum(ds.total_requests, 1e-9),
        0.0,
    )
    return OfflineObjective(
        di=float(np.mean(di_per_row[active])),
        art=float(np.mean(art_per_row[active])),
    )


def _get_dataset_backend_view(ds: OfflineDataset, xp_module):
    backend_key = "gpu" if (cp is not None and xp_module is cp) else "cpu"
    cache = getattr(_get_dataset_backend_view, "_cache", {})
    key = (id(ds), backend_key)
    if key not in cache:
        total_requests = xp_module.asarray(ds.total_requests, dtype=xp_module.float64)
        cache[key] = {
            "total_requests": total_requests,
            "node1_cpu_norm": xp_module.asarray(ds.node1_cpu_norm, dtype=xp_module.float64),
            "node2_cpu_norm": xp_module.asarray(ds.node2_cpu_norm, dtype=xp_module.float64),
            "node1_cpu_capacity": xp_module.asarray(ds.node1_cpu_capacity, dtype=xp_module.float64),
            "node2_cpu_capacity": xp_module.asarray(ds.node2_cpu_capacity, dtype=xp_module.float64),
            "node1_queue": xp_module.asarray(ds.node1_queue, dtype=xp_module.float64),
            "node2_queue": xp_module.asarray(ds.node2_queue, dtype=xp_module.float64),
            "node1_response_ms": xp_module.asarray(ds.node1_response_ms, dtype=xp_module.float64),
            "node2_response_ms": xp_module.asarray(ds.node2_response_ms, dtype=xp_module.float64),
            "active_mask": (total_requests > 0).astype(xp_module.float64),
            "active_count": float(max(ds.used_samples, 1)),
        }
        _get_dataset_backend_view._cache = cache
    return cache[key]


def evaluate_dataset_di_art_batch(
    params_batch: np.ndarray,
    ds: OfflineDataset,
    constants: Optional[SystemConstants] = None,
    use_gpu: Optional[bool] = None,
) -> tuple[np.ndarray, np.ndarray]:
    params_arr = np.asarray(params_batch, dtype=np.float64)
    if params_arr.ndim == 1:
        params_arr = params_arr[None, :]

    if ds.used_samples == 0:
        return (
            np.ones(params_arr.shape[0], dtype=np.float64),
            np.zeros(params_arr.shape[0], dtype=np.float64),
        )

    constants = extract_system_constants(ds) if constants is None else constants
    xp = get_xp(use_gpu)
    ds_backend = _get_dataset_backend_view(ds, xp)
    p = xp.asarray(params_arr, dtype=xp.float64)
    chunk_rows = GPU_TRAINING_CHUNK_ROWS if (cp is not None and xp is cp) else CPU_TRAINING_CHUNK_ROWS
    di_sum = xp.zeros(p.shape[0], dtype=xp.float64)
    art_sum = xp.zeros(p.shape[0], dtype=xp.float64)

    for start in range(0, ds.sample_count, chunk_rows):
        end = min(start + chunk_rows, ds.sample_count)

        node1_score = fuzzy_score_vectorized_batch(
            p,
            ds_backend["node1_cpu_norm"][start:end],
            ds_backend["node1_queue"][start:end],
            ds_backend["node1_response_ms"][start:end],
            xp_module=xp,
        )
        node2_score = fuzzy_score_vectorized_batch(
            p,
            ds_backend["node2_cpu_norm"][start:end],
            ds_backend["node2_queue"][start:end],
            ds_backend["node2_response_ms"][start:end],
            xp_module=xp,
        )

        total_requests = ds_backend["total_requests"][None, start:end]
        score_sum = node1_score + node2_score
        safe_sum = xp.where(score_sum > 1e-9, score_sum, 1.0)
        weight1 = xp.where(score_sum > 1e-9, node1_score / safe_sum, 0.5)
        weight2 = xp.where(score_sum > 1e-9, node2_score / safe_sum, 0.5)

        pred_req1 = weight1 * total_requests
        pred_req2 = weight2 * total_requests

        node1_cap = ds_backend["node1_cpu_capacity"][None, start:end]
        node2_cap = ds_backend["node2_cpu_capacity"][None, start:end]
        safe_cap1 = xp.where(node1_cap > 0.0, node1_cap, DEFAULT_NODE1_CPU_CAPACITY)
        safe_cap2 = xp.where(node2_cap > 0.0, node2_cap, DEFAULT_NODE2_CPU_CAPACITY)
        pred_cpu1_raw = xp.clip(constants.node1_cpu_base + (pred_req1 * constants.node1_cpu_cost), 0.0, safe_cap1)
        pred_cpu2_raw = xp.clip(constants.node2_cpu_base + (pred_req2 * constants.node2_cpu_cost), 0.0, safe_cap2)
        pred_cpu1 = xp.clip((pred_cpu1_raw / safe_cap1) * 100.0, 0.0, 100.0)
        pred_cpu2 = xp.clip((pred_cpu2_raw / safe_cap2) * 100.0, 0.0, 100.0)
        pred_rt1 = xp.maximum(constants.node1_rt_base + (pred_req1 * constants.node1_rt_cost), 0.0)
        pred_rt2 = xp.maximum(constants.node2_rt_base + (pred_req2 * constants.node2_rt_cost), 0.0)

        di_per_row = xp.abs(pred_cpu1 - pred_cpu2) / 100.0
        art_per_row = xp.where(
            total_requests > 0,
            ((pred_rt1 * pred_req1) + (pred_rt2 * pred_req2)) / xp.maximum(total_requests, 1e-9),
            0.0,
        )

        active_mask = ds_backend["active_mask"][None, start:end]
        di_sum += xp.sum(di_per_row * active_mask, axis=1)
        art_sum += xp.sum(art_per_row * active_mask, axis=1)

    active_count = ds_backend["active_count"]
    di_values = di_sum / active_count
    art_values = art_sum / active_count

    return (
        to_numpy(di_values).astype(np.float64, copy=False),
        to_numpy(art_values).astype(np.float64, copy=False),
    )


def build_objective_validation_trace(
    params: np.ndarray,
    ds: OfflineDataset,
    max_rows: int = 5,
    constants: Optional[SystemConstants] = None,
) -> Dict[str, object]:
    active = ds.total_requests > 0
    if not np.any(active):
        empty_constants = extract_system_constants(ds) if constants is None else constants
        return {
            "constants": asdict(empty_constants),
            "di": 1.0,
            "di_mean": 1.0,
            "art": 0.0,
            "art_mean": 0.0,
            "art_global_mean": 0.0,
            "mean_cpu_node1": 0.0,
            "mean_cpu_node2": 0.0,
            "mean_rt_node1": 0.0,
            "mean_rt_node2": 0.0,
            "mean_weight_node1": 0.5,
            "mean_weight_node2": 0.5,
            "mean_request_node1": 0.0,
            "mean_request_node2": 0.0,
            "sample_preview": [],
        }

    sim = _simulate_routing_response(params, ds, constants)
    constants = sim["constants"]

    di_value = float(np.mean(sim["di_per_row"][active]))
    art_value = float(np.mean(sim["art_per_row"][active]))
    mean_cpu_node1 = float(np.mean(sim["pred_cpu1"][active]))
    mean_cpu_node2 = float(np.mean(sim["pred_cpu2"][active]))
    mean_rt_node1 = float(np.mean(sim["pred_rt1"][active]))
    mean_rt_node2 = float(np.mean(sim["pred_rt2"][active]))
    mean_weight_node1 = float(np.mean(sim["weight1"][active]))
    mean_weight_node2 = float(np.mean(sim["weight2"][active]))
    mean_request_node1 = float(np.mean(sim["pred_req1"][active]))
    mean_request_node2 = float(np.mean(sim["pred_req2"][active]))

    active_idx = np.flatnonzero(active)
    preview_idx = active_idx[:max_rows]
    rows = []
    for i in preview_idx:
        rows.append(
            {
                "row": int(i),
                "total_requests": float(ds.total_requests[i]),
                "score_node1": float(sim["node1_score"][i]),
                "score_node2": float(sim["node2_score"][i]),
                "weight_node1": float(sim["weight1"][i]),
                "weight_node2": float(sim["weight2"][i]),
                "pred_node1_requests": float(sim["pred_req1"][i]),
                "pred_node2_requests": float(sim["pred_req2"][i]),
                "pred_node1_cpu_raw": float(sim["pred_cpu1_raw"][i]),
                "pred_node2_cpu_raw": float(sim["pred_cpu2_raw"][i]),
                "pred_node1_cpu_norm": float(sim["pred_cpu1"][i]),
                "pred_node2_cpu_norm": float(sim["pred_cpu2"][i]),
                "pred_node1_response_ms": float(sim["pred_rt1"][i]),
                "pred_node2_response_ms": float(sim["pred_rt2"][i]),
                "pred_di": float(sim["di_per_row"][i]),
                "pred_art": float(sim["art_per_row"][i]),
            }
        )

    return {
        "constants": asdict(constants),
        "di": di_value,
        "di_mean": di_value,
        "art": art_value,
        "art_mean": art_value,
        "art_global_mean": art_value,
        "mean_cpu_node1": mean_cpu_node1,
        "mean_cpu_node2": mean_cpu_node2,
        "mean_rt_node1": mean_rt_node1,
        "mean_rt_node2": mean_rt_node2,
        "mean_weight_node1": mean_weight_node1,
        "mean_weight_node2": mean_weight_node2,
        "mean_request_node1": mean_request_node1,
        "mean_request_node2": mean_request_node2,
        "sample_preview": rows,
    }


## CSV Loader (translasi loadOfflineSamples)


In [ ]:
# ----------------------------

def _find_header(index: Dict[str, str], *aliases: str) -> Optional[str]:
    for a in aliases:
        k = a.strip().lower()
        if k in index:
            return index[k]
    return None


def _require_header(index: Dict[str, str], *aliases: str) -> str:
    found = _find_header(index, *aliases)
    if found is None:
        raise ValueError(f"Kolom wajib tidak ditemukan. Alias dicari: {aliases}")
    return found


def _to_numeric_strict(df: pd.DataFrame, col: str, kind: str = "float") -> np.ndarray:
    try:
        series = pd.to_numeric(df[col], errors="raise")
    except Exception as exc:
        raise ValueError(f"Gagal parse numeric kolom '{col}': {exc}") from exc

    if kind == "int":
        return series.astype("int64").to_numpy()
    return series.astype("float64").to_numpy()


def _sanitize_float_array(values: np.ndarray, fallback: float = 0.0) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float64)
    if np.all(np.isfinite(arr)):
        return arr
    finite = arr[np.isfinite(arr)]
    fill = float(np.median(finite)) if finite.size > 0 else float(fallback)
    return np.nan_to_num(arr, nan=fill, posinf=fill, neginf=fill)


def _fallback_from_series(series: pd.Series, default: float = 0.0) -> float:
    finite = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if finite.empty:
        return float(default)
    return float(finite.median())


def _strip_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(col).strip() for col in out.columns]
    object_cols = out.select_dtypes(include=["object"]).columns
    for col in object_cols:
        out[col] = out[col].map(lambda value: value.strip() if isinstance(value, str) else value)
    return out


def _clean_numeric_series(
    series: pd.Series,
    *,
    fallback: float,
    lower: Optional[float] = None,
    upper: Optional[float] = None,
    zero_as_missing: bool = False,
    fill_strategy: str = "median",
) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if zero_as_missing:
        numeric = numeric.mask(numeric <= 0)

    if fill_strategy == "ffill_bfill":
        fallback_value = _fallback_from_series(numeric, fallback)
        numeric = numeric.ffill().bfill().fillna(fallback_value)
    else:
        fallback_value = _fallback_from_series(numeric, fallback)
        numeric = numeric.fillna(fallback_value)

    if lower is not None or upper is not None:
        numeric = numeric.clip(lower=lower, upper=upper)
    return numeric.astype("float64")


def _clean_offline_dataframe(
    df_raw: pd.DataFrame,
    *,
    total_req_col: str,
    req1_col: str,
    req2_col: str,
    cpu1_col: str,
    cpu2_col: str,
    q1_col: str,
    q2_col: str,
    r1_col: str,
    r2_col: str,
    ts_col: Optional[str],
    extra_cols: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    df = _strip_object_columns(df_raw)

    selected = [total_req_col, req1_col, req2_col, cpu1_col, cpu2_col, q1_col, q2_col, r1_col, r2_col]
    if ts_col is not None:
        selected.append(ts_col)
    if extra_cols is not None:
        selected.extend([col for col in extra_cols if col is not None])
    selected = list(dict.fromkeys(selected))
    df = df[selected].copy()
    df = df.dropna(how="all")

    if ts_col is not None:
        parsed_ts = pd.to_datetime(df[ts_col], errors="coerce", utc=True)
        order = np.argsort(parsed_ts.fillna(pd.Timestamp.max.tz_localize("UTC")).to_numpy())
        df = df.iloc[order].copy()
        df = df.drop_duplicates(subset=[ts_col], keep="last")
    else:
        df = df.drop_duplicates(keep="last")

    req1 = pd.to_numeric(df[req1_col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    req2 = pd.to_numeric(df[req2_col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    total_req = pd.to_numeric(df[total_req_col], errors="coerce").replace([np.inf, -np.inf], np.nan)

    req1 = req1.clip(lower=0.0)
    req2 = req2.clip(lower=0.0)
    total_req = total_req.clip(lower=0.0)
    req_sum = req1 + req2
    total_req = total_req.fillna(req_sum)
    total_req = np.maximum(total_req.to_numpy(dtype=np.float64), req_sum.to_numpy(dtype=np.float64))

    df[req1_col] = np.rint(req1.to_numpy(dtype=np.float64)).astype(np.int64)
    df[req2_col] = np.rint(req2.to_numpy(dtype=np.float64)).astype(np.int64)
    df[total_req_col] = total_req

    active_mask = df[total_req_col].to_numpy(dtype=np.float64) > 0.0
    df = df.loc[active_mask].copy()
    if df.empty:
        raise ValueError("dataset tidak memiliki row traffic aktif (total_requests > 0)")

    df[cpu1_col] = _clean_numeric_series(df[cpu1_col], fallback=0.0, lower=0.0, upper=100.0)
    df[cpu2_col] = _clean_numeric_series(df[cpu2_col], fallback=0.0, lower=0.0, upper=100.0)
    df[q1_col] = _clean_numeric_series(df[q1_col], fallback=0.0, lower=0.0)
    df[q2_col] = _clean_numeric_series(df[q2_col], fallback=0.0, lower=0.0)
    df[r1_col] = _clean_numeric_series(df[r1_col], fallback=1.0, lower=0.0, zero_as_missing=True, fill_strategy="ffill_bfill")
    df[r2_col] = _clean_numeric_series(df[r2_col], fallback=1.0, lower=0.0, zero_as_missing=True, fill_strategy="ffill_bfill")

    return df.reset_index(drop=True)


def load_offline_samples_csv(csv_path: str | Path) -> OfflineDataset:
    csv_path = Path(csv_path)
    df_raw = pd.read_csv(csv_path, sep=None, engine="python", on_bad_lines="warn")
    if df_raw.empty:
        raise ValueError(f"Dataset kosong: {csv_path}")

    df_raw = _strip_object_columns(df_raw)
    col_index = {c.strip().lower(): c for c in df_raw.columns}

    total_req_col = _require_header(col_index, "total_requests")
    req1_col = _require_header(col_index, "node1_requests")
    req2_col = _require_header(col_index, "node2_requests")
    cpu1_norm_col = _find_header(col_index, "node1_cpu_normalized_usage", "node1_cpu_usage_normalized")
    cpu2_norm_col = _find_header(col_index, "node2_cpu_normalized_usage", "node2_cpu_usage_normalized")
    cpu1_raw_col = _find_header(col_index, "node1_cpu_raw_usage", "node1_cpu_usage_raw")
    cpu2_raw_col = _find_header(col_index, "node2_cpu_raw_usage", "node2_cpu_usage_raw")
    if cpu1_norm_col is None and cpu1_raw_col is None:
        raise ValueError("header CPU node1 tidak ditemukan (butuh kolom raw atau normalized)")
    if cpu2_norm_col is None and cpu2_raw_col is None:
        raise ValueError("header CPU node2 tidak ditemukan (butuh kolom raw atau normalized)")
    cap1_col = _find_header(col_index, "node1_cpu_capacity")
    cap2_col = _find_header(col_index, "node2_cpu_capacity")
    q1_col = _require_header(col_index, "node1_queue", "node1_inflight")
    q2_col = _require_header(col_index, "node2_queue", "node2_inflight")
    r1_col = _require_header(col_index, "node1_response_ms", "node1_latency_ms")
    r2_col = _require_header(col_index, "node2_response_ms", "node2_latency_ms")
    ts_col = _find_header(col_index, "timestamp_utc", "timestamp")
    cpu1_col = cpu1_norm_col or cpu1_raw_col
    cpu2_col = cpu2_norm_col or cpu2_raw_col

    df = _clean_offline_dataframe(
        df_raw,
        total_req_col=total_req_col,
        req1_col=req1_col,
        req2_col=req2_col,
        cpu1_col=cpu1_col,
        cpu2_col=cpu2_col,
        q1_col=q1_col,
        q2_col=q2_col,
        r1_col=r1_col,
        r2_col=r2_col,
        ts_col=ts_col,
        extra_cols=[cpu1_norm_col, cpu2_norm_col, cpu1_raw_col, cpu2_raw_col, cap1_col, cap2_col],
    )

    req1 = _to_numeric_strict(df, req1_col, "int")
    req2 = _to_numeric_strict(df, req2_col, "int")
    total_req = _sanitize_float_array(pd.to_numeric(df[total_req_col], errors="coerce").to_numpy(), 0.0)

    cap1 = (
        _sanitize_float_array(pd.to_numeric(df[cap1_col], errors="coerce").to_numpy(), DEFAULT_NODE1_CPU_CAPACITY)
        if cap1_col is not None
        else np.full(req1.shape[0], DEFAULT_NODE1_CPU_CAPACITY, dtype=np.float64)
    )
    cap2 = (
        _sanitize_float_array(pd.to_numeric(df[cap2_col], errors="coerce").to_numpy(), DEFAULT_NODE2_CPU_CAPACITY)
        if cap2_col is not None
        else np.full(req2.shape[0], DEFAULT_NODE2_CPU_CAPACITY, dtype=np.float64)
    )
    cap1 = np.where(cap1 > 0.0, cap1, DEFAULT_NODE1_CPU_CAPACITY)
    cap2 = np.where(cap2 > 0.0, cap2, DEFAULT_NODE2_CPU_CAPACITY)

    if cpu1_raw_col is not None:
        cpu1_raw = _sanitize_float_array(pd.to_numeric(df[cpu1_raw_col], errors="coerce").to_numpy(), 0.0)
        cpu1_raw = np.clip(cpu1_raw, 0.0, cap1)
    else:
        cpu1_norm_source = _sanitize_float_array(pd.to_numeric(df[cpu1_norm_col], errors="coerce").to_numpy(), 0.0)
        cpu1_raw = np.clip((np.clip(cpu1_norm_source, 0.0, 100.0) / 100.0) * cap1, 0.0, cap1)

    if cpu2_raw_col is not None:
        cpu2_raw = _sanitize_float_array(pd.to_numeric(df[cpu2_raw_col], errors="coerce").to_numpy(), 0.0)
        cpu2_raw = np.clip(cpu2_raw, 0.0, cap2)
    else:
        cpu2_norm_source = _sanitize_float_array(pd.to_numeric(df[cpu2_norm_col], errors="coerce").to_numpy(), 0.0)
        cpu2_raw = np.clip((np.clip(cpu2_norm_source, 0.0, 100.0) / 100.0) * cap2, 0.0, cap2)

    if cpu1_norm_col is not None:
        cpu1 = _sanitize_float_array(pd.to_numeric(df[cpu1_norm_col], errors="coerce").to_numpy(), 0.0)
        cpu1 = np.clip(cpu1, 0.0, 100.0)
    else:
        cpu1 = to_normalized_cpu(cpu1_raw, cap1)

    if cpu2_norm_col is not None:
        cpu2 = _sanitize_float_array(pd.to_numeric(df[cpu2_norm_col], errors="coerce").to_numpy(), 0.0)
        cpu2 = np.clip(cpu2, 0.0, 100.0)
    else:
        cpu2 = to_normalized_cpu(cpu2_raw, cap2)

    q1 = _sanitize_float_array(pd.to_numeric(df[q1_col], errors="coerce").to_numpy(), 0.0)
    q2 = _sanitize_float_array(pd.to_numeric(df[q2_col], errors="coerce").to_numpy(), 0.0)
    q1 = np.maximum(q1, 0.0)
    q2 = np.maximum(q2, 0.0)

    r1 = _sanitize_float_array(pd.to_numeric(df[r1_col], errors="coerce").to_numpy(), _fallback_from_series(df[r1_col], 1.0))
    r2 = _sanitize_float_array(pd.to_numeric(df[r2_col], errors="coerce").to_numpy(), _fallback_from_series(df[r2_col], 1.0))
    r1 = np.maximum(r1, 0.0)
    r2 = np.maximum(r2, 0.0)

    if ts_col is not None:
        ts = df[ts_col].astype(str).fillna("").to_numpy()
    else:
        ts = np.array([""] * req1.shape[0], dtype=object)

    if req1.shape[0] == 0:
        raise ValueError("dataset tidak memiliki row data")

    scenario = csv_path.stem
    if scenario.startswith("trace_"):
        scenario = scenario.replace("trace_", "", 1)

    return OfflineDataset(
        scenario_name=scenario,
        timestamp=ts,
        total_requests=np.maximum(total_req, 0.0),
        node1_requests=req1,
        node2_requests=req2,
        node1_cpu_raw=cpu1_raw,
        node2_cpu_raw=cpu2_raw,
        node1_cpu_norm=cpu1,
        node2_cpu_norm=cpu2,
        node1_cpu_capacity=cap1,
        node2_cpu_capacity=cap2,
        node1_queue=q1,
        node2_queue=q2,
        node1_response_ms=r1,
        node2_response_ms=r2,
    )


def load_base_params(base_json_path: Optional[str | Path] = None) -> np.ndarray:
    if base_json_path is None:
        return DEFAULT_BASE_PARAMS.copy()

    p = Path(base_json_path)
    if not p.exists():
        return DEFAULT_BASE_PARAMS.copy()

    with p.open("r", encoding="utf-8") as f:
        arr = json.load(f)

    arr_np = np.asarray(arr, dtype=np.float64)
    if arr_np.shape[0] != DIMENSIONS:
        raise ValueError(f"panjang base params harus {DIMENSIONS}, dapat {arr_np.shape[0]}")
    return arr_np


def build_dataset_quality_report(ds: OfflineDataset) -> Dict[str, object]:
    def stats(arr: np.ndarray) -> Dict[str, float]:
        arr = np.asarray(arr, dtype=np.float64)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            return {"finite_count": 0.0, "nan_count": float(arr.size), "min": float("nan"), "max": float("nan"), "mean": float("nan")}
        return {
            "finite_count": float(finite.size),
            "nan_count": float(arr.size - finite.size),
            "min": float(np.min(finite)),
            "max": float(np.max(finite)),
            "mean": float(np.mean(finite)),
        }

    return {
        "scenario": ds.scenario_name,
        "sample_count": ds.sample_count,
        "used_samples": ds.used_samples,
        "node1_response_ms": stats(ds.node1_response_ms),
        "node2_response_ms": stats(ds.node2_response_ms),
        "node1_cpu_normalized": stats(ds.node1_cpu_norm),
        "node2_cpu_normalized": stats(ds.node2_cpu_norm),
        "node1_queue": stats(ds.node1_queue),
        "node2_queue": stats(ds.node2_queue),
        "node1_requests": stats(ds.node1_requests.astype(np.float64)),
        "node2_requests": stats(ds.node2_requests.astype(np.float64)),
    }


## MOPSO Offline (translasi OptimizeOffline)


In [ ]:
# ----------------------------

def add_to_archive(archive: List[OfflineSolution], candidate: OfflineSolution) -> List[OfflineSolution]:
    # ANTI-CLONE FILTER: Tolak kembaran biar arsip tidak penuh kloningan
    for sol in archive:
        if abs(sol.objective.di - candidate.objective.di) < 1e-5 and abs(sol.objective.art - candidate.objective.art) < 1e-3:
            return archive  # Tolak kandidat yang nilainya kembar
            
    keep: List[OfflineSolution] = []
    for i, sol in enumerate(archive):
        if dominates(sol.objective, candidate.objective):
            keep.extend(archive[i:])
            return keep
        if not dominates(candidate.objective, sol.objective):
            keep.append(sol)

    keep.append(candidate)
    if len(keep) <= MAX_ARCHIVE:
        return keep

    keep.sort(key=lambda s: (s.objective.di, s.objective.art))
    return keep[:MAX_ARCHIVE]


def choose_sane_candidate(result: OfflineResult) -> Optional[OfflineSolution]:
    best = result.best_balanced
    if is_sane_params(np.asarray(best.params, dtype=np.float64)):
        return best

    for sol in result.archive:
        if is_sane_params(np.asarray(sol.params, dtype=np.float64)):
            return sol
    return None


def plot_convergence(
    history: Sequence[float],
    scenario_name: str,
    out_dir: str | Path,
    iter_best_history: Optional[Sequence[float]] = None,
    iter_mean_history: Optional[Sequence[float]] = None,
    archive_size_history: Optional[Sequence[float]] = None,
) -> Path:
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"convergence_trace_{scenario_name}.png"

    if len(history) == 0:
        return out_file

    x = np.arange(1, len(history) + 1)
    y = np.asarray(history, dtype=np.float64)
    y[~np.isfinite(y)] = np.nan

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, y, color="#1f77b4", linewidth=2.2, label="Best-So-Far Objective")

    if iter_best_history is not None and len(iter_best_history) == len(history):
        iter_best = np.asarray(iter_best_history, dtype=np.float64)
        iter_best[~np.isfinite(iter_best)] = np.nan
        ax.plot(x, iter_best, color="#ff7f0e", linewidth=1.6, alpha=0.9, label="Iteration Best Objective")

    if iter_mean_history is not None and len(iter_mean_history) == len(history):
        iter_mean = np.asarray(iter_mean_history, dtype=np.float64)
        iter_mean[~np.isfinite(iter_mean)] = np.nan
        ax.plot(x, iter_mean, color="#2ca02c", linewidth=1.4, alpha=0.85, label="Iteration Mean Objective")

    ax2 = None
    if archive_size_history is not None and len(archive_size_history) == len(history):
        archive_sizes = np.asarray(archive_size_history, dtype=np.float64)
        archive_sizes[~np.isfinite(archive_sizes)] = np.nan
        ax2 = ax.twinx()
        ax2.plot(x, archive_sizes, color="#7f7f7f", linewidth=1.2, linestyle="--", alpha=0.7, label="Archive Size")
        ax2.set_ylabel("Archive Size")
    ax.set_title(f"Convergence Trace (DI-first Objective) - {scenario_name}")
    ax.set_xlabel("Iterasi")
    ax.set_ylabel("DI Ratio + epsilon * (ART / 1000.0)")
    ax.grid(True, linestyle="--", alpha=0.35)
    handles, labels = ax.get_legend_handles_labels()
    if ax2 is not None:
        handles2, labels2 = ax2.get_legend_handles_labels()
        handles.extend(handles2)
        labels.extend(labels2)
    ax.legend(handles, labels, loc="best")
    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def plot_pareto_archive(archive_rows: Sequence[ArchiveScoreRow], selected: Optional[ArchiveScoreRow], scenario_name: str, out_dir: str | Path) -> Path:
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"pareto_archive_{scenario_name}.png"
    if len(archive_rows) == 0:
        return out_file

    di = np.array([r.di for r in archive_rows], dtype=np.float64)
    art = np.array([r.art for r in archive_rows], dtype=np.float64)
    score = np.array([r.score for r in archive_rows], dtype=np.float64)

    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(di, art, c=score, cmap="viridis", s=42, alpha=0.85, edgecolors="none")
    ax.set_title(f"Pareto Archive - {scenario_name}")
    ax.set_xlabel("DI Ratio (Capacity-Normalized CPU)")
    ax.set_ylabel("ART Mean RT (ms)")
    ax.grid(True, linestyle="--", alpha=0.3)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Selection Score (DI-first)")

    if selected is not None:
        ax.scatter([selected.di], [selected.art], s=180, marker="*", color="#d62728", label="Selected")
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def plot_archive_normalization(archive_rows: Sequence[ArchiveScoreRow], selected: Optional[ArchiveScoreRow], scenario_name: str, out_dir: str | Path, top_n: int = 10) -> Path:
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"normalized_archive_{scenario_name}.png"
    if len(archive_rows) == 0:
        return out_file

    ordered = sorted(archive_rows, key=lambda r: (r.di, r.art, r.score))[:top_n]
    labels = [f"S{r.index + 1}" for r in ordered]
    x = np.arange(len(ordered))
    width = 0.28

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.bar(x - width, [r.norm_di for r in ordered], width=width, label="Norm DI Ratio", color="#1f77b4")
    ax1.bar(x, [r.norm_art for r in ordered], width=width, label="Norm ART", color="#ff7f0e")
    ax1.bar(x + width, [r.score for r in ordered], width=width, label="Selection Score", color="#2ca02c")
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.set_ylabel("Normalized Ranking Value")
    ax1.set_title(f"Top Archive Candidates (DI-first Ranking) - {scenario_name}")
    ax1.grid(True, axis="y", linestyle="--", alpha=0.3)

    y_min, y_max = 0.0, 1.05
    if selected is not None:
        ax1.axhline(selected.score, color="#d62728", linestyle=":", linewidth=1.5, label="Selected Selection Score")
        y_min = min(y_min, selected.score - 0.05)
        y_max = max(y_max, selected.score + 0.05)

    ax1.set_ylim(y_min, y_max)

    ax1.legend(loc="upper left")

    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def plot_objective_comparison(base_obj: OfflineObjective, opt_obj: OfflineObjective, scenario_name: str, out_dir: str | Path) -> Path:
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"objective_comparison_{scenario_name}.png"

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    panels = [
        (axes[0], "DI Ratio (Capacity-Normalized CPU)", base_obj.di, opt_obj.di, "#7f7f7f", "#1f77b4"),
        (axes[1], "ART Mean RT (ms)", base_obj.art, opt_obj.art, "#7f7f7f", "#ff7f0e"),
    ]

    for ax, title, base_val, opt_val, c1, c2 in panels:
        bars = ax.bar(["Base", "Optimized"], [base_val, opt_val], color=[c1, c2], width=0.55)
        ax.set_title(title)
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2.0, height, f"{height:.3f}", ha="center", va="bottom", fontsize=9)

    fig.suptitle(f"Base vs Optimized - {scenario_name}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def plot_dataset_overview(ds: OfflineDataset, scenario_name: str, out_dir: str | Path) -> Path:
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_file = out_dir_path / f"dataset_overview_{scenario_name}.png"

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()

    axes[0].plot(ds.node1_response_ms[:250], color="#1f77b4", linewidth=1.5)
    axes[0].set_title("Node1 Response Time (sample)")
    axes[0].set_ylabel("ms")
    axes[0].grid(True, linestyle="--", alpha=0.25)

    axes[1].plot(ds.node2_response_ms[:250], color="#ff7f0e", linewidth=1.5)
    axes[1].set_title("Node2 Response Time (sample)")
    axes[1].set_ylabel("ms")
    axes[1].grid(True, linestyle="--", alpha=0.25)

    axes[2].hist(ds.node1_response_ms, bins=40, color="#1f77b4", alpha=0.8)
    axes[2].set_title("Node1 Response Distribution")
    axes[2].set_xlabel("ms")
    axes[2].set_ylabel("count")
    axes[2].grid(True, linestyle="--", alpha=0.25)

    axes[3].hist(ds.node2_response_ms, bins=40, color="#ff7f0e", alpha=0.8)
    axes[3].set_title("Node2 Response Distribution")
    axes[3].set_xlabel("ms")
    axes[3].set_ylabel("count")
    axes[3].grid(True, linestyle="--", alpha=0.25)

    fig.suptitle(f"Dataset Overview - {scenario_name}")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def optimize_offline(ds: OfflineDataset, base_params: np.ndarray, cfg: OfflineConfig) -> OfflineResult:
    cfg = cfg.normalized()
    base_params = repair_params(base_params)

    if base_params.shape[0] != DIMENSIONS:
        raise ValueError("base params harus berukuran 27")
    if ds.sample_count == 0:
        raise ValueError("dataset kosong")
    if ds.used_samples == 0:
        raise ValueError("dataset tidak punya sample dengan total request > 0")

    system_constants = extract_system_constants(ds)

    rng = np.random.default_rng(cfg.seed)

    spread_vector = np.full(DIMENSIONS, float(cfg.initial_spread), dtype=np.float64)
    spread_vector[9:] *= 10.0
    x = base_params[None, :] + rng.uniform(-spread_vector[None, :], spread_vector[None, :], size=(cfg.particles, DIMENSIONS))
    x = np.clip(x, LOWER_BOUNDS[None, :], UPPER_BOUNDS[None, :]).astype(np.float64)
    v = rng.uniform(-spread_vector[None, :], spread_vector[None, :], size=(cfg.particles, DIMENSIONS)).astype(np.float64)
    pbest = np.empty_like(x)
    pbest_di = np.full(cfg.particles, np.inf, dtype=np.float64)
    pbest_art = np.full(cfg.particles, np.inf, dtype=np.float64)

    for i in range(cfg.particles):
        x[i] = repair_params(x[i])
    pbest[:] = x

    archive: List[OfflineSolution] = []
    score_history: List[float] = []
    iter_best_history: List[float] = []
    iter_mean_history: List[float] = []
    archive_size_history: List[float] = []
    best_score_so_far = float("inf")
    stagnation_counter = 0
    stagnation_events = 0

    iter_progress = tqdm(range(cfg.iterations), desc=f"Optimize[{ds.scenario_name}]", unit="iter", leave=False)
    for t in iter_progress:
        w = INERTIA_MAX_W - (INERTIA_MAX_W - INERTIA_MIN_W) * (float(t) / float(max_int(cfg.iterations - 1, 1)))

        di_values, art_values = evaluate_dataset_di_art_batch(x, ds, system_constants, use_gpu=None)
        current_scores = di_values + ((art_values / 1000.0) * DI_ART_TIEBREAK_WEIGHT)
        old_scores = pbest_di + ((pbest_art / 1000.0) * DI_ART_TIEBREAK_WEIGHT)
        dominates_new = ((di_values <= pbest_di) & (art_values <= pbest_art) & ((di_values < pbest_di) | (art_values < pbest_art)))
        dominates_old = ((pbest_di <= di_values) & (pbest_art <= art_values) & ((pbest_di < di_values) | (pbest_art < art_values)))
        replace_mask = np.isinf(pbest_di) | dominates_new | ((~dominates_old) & (current_scores < old_scores))
        pbest[replace_mask] = x[replace_mask]
        pbest_di[replace_mask] = di_values[replace_mask]
        pbest_art[replace_mask] = art_values[replace_mask]

        iter_best_current = float(np.min(current_scores)) if current_scores.size > 0 else float("inf")
        iter_mean_current = float(np.mean(current_scores)) if current_scores.size > 0 else float("inf")

        for i in range(cfg.particles):
            obj = OfflineObjective(di=float(di_values[i]), art=float(art_values[i]))
            cand = OfflineSolution(params=x[i].tolist(), objective=obj)
            archive = add_to_archive(archive, cand)

        if not archive:
            score_history.append(best_score_so_far)
            iter_best_history.append(iter_best_current)
            iter_mean_history.append(iter_mean_current)
            archive_size_history.append(0.0)
            continue

        iter_best_abs = min(objective_score(sol.objective) for sol in archive)
        if iter_best_abs < best_score_so_far:
            best_score_so_far = iter_best_abs
            stagnation_counter = 0
        else:
            stagnation_counter += 1
        score_history.append(best_score_so_far)
        iter_best_history.append(iter_best_current)
        iter_mean_history.append(iter_mean_current)
        archive_size_history.append(float(len(archive)))
        try:
            iter_progress.set_postfix(best_abs=f"{best_score_so_far:.6f}", archive=len(archive), stagnation=stagnation_counter)
        except Exception:
            pass

        leader_indices = rng.integers(0, len(archive), size=cfg.particles)
        leader_params = np.asarray([archive[int(idx)].params for idx in leader_indices], dtype=np.float64)
        r1 = rng.uniform(0.0, 1.0, size=(cfg.particles, DIMENSIONS))
        r2 = rng.uniform(0.0, 1.0, size=(cfg.particles, DIMENSIONS))
        v = (
            (w * v)
            + (COGNITIVE_C1 * r1 * (pbest - x))
            + (SOCIAL_C2 * r2 * (leader_params - x))
        )
        x = np.clip(x + v, LOWER_BOUNDS[None, :], UPPER_BOUNDS[None, :])

        for i in range(cfg.particles):
            x[i] = repair_params(x[i])

        if stagnation_counter >= STAGNATION_PATIENCE:
            reseed_count = max(1, int(np.ceil(cfg.particles * RESEED_FRACTION)))
            worst_indices = np.argsort(current_scores)[-reseed_count:]
            reseed_scale = spread_vector * RESEED_SCALE_MULTIPLIER

            if len(archive) > 0:
                archive_pick = rng.integers(0, len(archive), size=reseed_count)
                archive_anchors = np.asarray([archive[int(idx)].params for idx in archive_pick], dtype=np.float64)
                use_archive = rng.uniform(0.0, 1.0, size=(reseed_count, 1)) < 0.5
                anchors = np.where(use_archive, archive_anchors, base_params[None, :])
            else:
                anchors = np.repeat(base_params[None, :], reseed_count, axis=0)

            x[worst_indices] = anchors + rng.uniform(-reseed_scale[None, :], reseed_scale[None, :], size=(reseed_count, DIMENSIONS))
            x[worst_indices] = np.clip(x[worst_indices], LOWER_BOUNDS[None, :], UPPER_BOUNDS[None, :]).astype(np.float64)
            v[worst_indices] = rng.uniform(-reseed_scale[None, :], reseed_scale[None, :], size=(reseed_count, DIMENSIONS)).astype(np.float64)
            for idx in worst_indices:
                x[idx] = repair_params(x[idx])
            pbest[worst_indices] = x[worst_indices]
            pbest_di[worst_indices] = np.inf
            pbest_art[worst_indices] = np.inf
            stagnation_counter = 0
            stagnation_events += 1


    archive.sort(key=lambda s: (s.objective.di, s.objective.art))

    best_balanced, best_balanced_score, archive_trace, normalization_stats = select_best_balanced_solution(archive)
    best_di = min(archive, key=lambda s: s.objective.di)
    best_art = min(archive, key=lambda s: s.objective.art)

    return OfflineResult(
        generated_at=time.strftime("%Y-%m-%dT%H:%M:%S"),
        sample_count=ds.sample_count,
        used_samples=ds.used_samples,
        config=cfg,
        archive=archive,
        best_balanced=best_balanced,
        best_di=best_di,
        best_art=best_art,
        best_balanced_score=best_balanced_score,
        system_constants=system_constants,
        archive_trace=archive_trace,
        normalization_stats=normalization_stats,
        score_history=score_history,
        iter_best_history=iter_best_history,
        iter_mean_history=iter_mean_history,
        archive_size_history=archive_size_history,
        stagnation_events=stagnation_events,
    )


## Batch per skenario (isolated)


In [ ]:
# ----------------------------

def run_scenario_optimization(
    csv_path: str | Path,
    base_params: np.ndarray,
    out_dir: str | Path,
    particles: int = 30,
    iterations: int = 3000,
    spread: float = 8.0,
    seed: int = 0,
    runs: int = 5,
    allow_regression: bool = False,
) -> Dict:
    ds = load_offline_samples_csv(csv_path)

    total_runs = runs if runs > 0 else 1
    base_seed = seed if seed != 0 else int(time.time_ns())

    summaries: List[Dict] = []
    best_result: Optional[OfflineResult] = None
    best_score = float("inf")
    best_run_meta: Optional[Dict] = None

    run_progress = tqdm(range(total_runs), desc=f"Runs[{ds.scenario_name}]", unit="run", leave=False)
    for i in run_progress:
        run_seed = base_seed + i
        cfg = OfflineConfig(particles=particles, iterations=iterations, initial_spread=spread, seed=run_seed)

        try:
            result = optimize_offline(ds, base_params, cfg)
            score = result.best_balanced_score
            summaries.append(
                {
                    "run_index": i + 1,
                    "seed": int(run_seed),
                    "score_balanced": float(score),
                    "error": None,
                }
            )

            if best_result is None or score < best_score:
                best_result = result
                best_score = score
                best_run_meta = summaries[-1]
            try:
                run_progress.set_postfix(best_bal=f"{best_score:.6f}")
            except Exception:
                pass

        except Exception as exc:
            summaries.append(
                {
                    "run_index": i + 1,
                    "seed": int(run_seed),
                    "score_balanced": float("inf"),
                    "error": repr(exc),
                }
            )

    if best_result is None:
        failure_details = "\n".join(
            f"- run {item['run_index']} seed={item['seed']}: {item['error']}"
            for item in summaries
            if item.get("error")
        )
        raise RuntimeError(f"Semua run gagal untuk skenario {ds.scenario_name}\n{failure_details}")

    sane_candidate = choose_sane_candidate(best_result)
    if sane_candidate is None:
        repaired = repair_params(np.asarray(best_result.best_balanced.params, dtype=np.float64))
        if not is_sane_params(repaired):
            raise RuntimeError("Tidak ada kandidat sane di semua run")
        best_result.best_balanced = OfflineSolution(params=repaired.tolist(), objective=best_result.best_balanced.objective)
    else:
        best_result.best_balanced = sane_candidate

    # evaluasi base vs optimized
    system_constants = best_result.system_constants
    base_obj = evaluate_historical_base_objective(ds)
    opt_obj = evaluate_dataset_di_art(np.asarray(best_result.best_balanced.params, dtype=np.float64), ds, system_constants)
    best_result.best_balanced.objective = opt_obj

    selected_row = find_trace_row(best_result.archive_trace, best_result.best_balanced.params)
    stats = best_result.normalization_stats
    if stats:
        base_norm_di = minmax_scale_value(base_obj.di, stats["min_di"], stats["max_di"])
        base_norm_art = minmax_scale_value(base_obj.art, stats["min_art"], stats["max_art"])
        base_score = balanced_score_from_normalized(base_norm_di, base_norm_art)
        opt_norm_di = minmax_scale_value(opt_obj.di, stats["min_di"], stats["max_di"])
        opt_norm_art = minmax_scale_value(opt_obj.art, stats["min_art"], stats["max_art"])
        opt_score_from_objective = balanced_score_from_normalized(opt_norm_di, opt_norm_art)
    else:
        base_score = 0.0
        opt_score_from_objective = 0.0
    opt_score = opt_score_from_objective
    if selected_row is None and stats:
        selected_row = ArchiveScoreRow(
            index=-1,
            params=best_result.best_balanced.params,
            di=float(opt_obj.di),
            art=float(opt_obj.art),
            norm_di=float(opt_norm_di),
            norm_art=float(opt_norm_art),
            score=float(opt_score),
        )
    best_result.best_balanced_score = opt_score
    if best_run_meta is not None:
        best_run_meta["score_balanced"] = float(opt_score)

    if (not allow_regression) and (opt_score >= base_score):
        raise RuntimeError(
            f"Optimized tidak mengalahkan base pada skenario {ds.scenario_name} "
            f"(base={base_score:.6f}, opt={opt_score:.6f})"
        )

    scenario_name = ds.scenario_name
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    convergence_out = plot_convergence(
        best_result.score_history,
        scenario_name,
        out_dir,
        iter_best_history=best_result.iter_best_history,
        iter_mean_history=best_result.iter_mean_history,
        archive_size_history=best_result.archive_size_history,
    )
    pareto_out = plot_pareto_archive(best_result.archive_trace, selected_row, scenario_name, out_dir)
    normalized_out = plot_archive_normalization(best_result.archive_trace, selected_row, scenario_name, out_dir)
    comparison_out = plot_objective_comparison(base_obj, opt_obj, scenario_name, out_dir)

    params_out = out_dir / f"opt_fuzzy_{scenario_name}.json"
    report_out = out_dir / f"mopso_report_{scenario_name}.json"

    with params_out.open("w", encoding="utf-8") as f:
        json.dump(best_result.best_balanced.params, f, ensure_ascii=False, indent=2)

    trace_map = {tuple(row.params): row for row in best_result.archive_trace}
    archive_export = []
    for sol in best_result.archive:
        row = trace_map.get(tuple(sol.params))
        archive_export.append(
            {
                "params": sol.params,
                "objective": {"di": sol.objective.di, "art": sol.objective.art, "max_rt_p99": sol.objective.art},
                "score_balanced": row.score if row is not None else None,
            }
        )

    validation_steps = {
        "step_1_dataset_summary": {
            "scenario": scenario_name,
            "sample_count": ds.sample_count,
            "used_samples": ds.used_samples,
            "archive_size": len(best_result.archive),
            "training_backend": current_training_backend(),
            "gpu_available": GPU_AVAILABLE,
        },
        "step_2_objective_reference": {
            "routing_formula": "weight_i = score_i / (score_node1 + score_node2)",
            "request_formula": "pred_req_i = total_requests * weight_i",
            "cpu_model": "pred_cpu_raw_i = base_cpu_i + (pred_req_i * cost_cpu_i)",
            "cpu_normalization": "pred_cpu_i = to_normalized_cpu(pred_cpu_raw_i, capacity_i)",
            "rt_model": "pred_rt_i = base_rt_i + (pred_req_i * cost_rt_i)",
            "di_formula": "mean(abs(pred_cpu_node1_norm - pred_cpu_node2_norm) / 100.0)",
            "art_formula": "mean(((pred_rt_node1 * pred_req_node1) + (pred_rt_node2 * pred_req_node2)) / total_requests)",
            "base_di_formula": "mean(abs(node1_cpu_norm - node2_cpu_norm) / 100.0)",
            "base_art_formula": "mean(((node1_response_ms * node1_requests) + (node2_response_ms * node2_requests)) / total_requests)  # row-level weighted RT on active snapshots",
            "base_objective_source": "historical CSV ground truth (direct observed CPU/RT, no regression replay)",
            "dataset_filter": "total_requests > 0",
        },
        "step_3_base_trace": build_objective_validation_trace(base_params, ds, constants=system_constants),
        "step_4_optimized_trace": build_objective_validation_trace(np.asarray(best_result.best_balanced.params, dtype=np.float64), ds, constants=system_constants),
        "step_5_normalization": best_result.normalization_stats,
        "step_6_selected_solution": {
            "system_constants": asdict(system_constants),
            "params": best_result.best_balanced.params,
            "di": float(best_result.best_balanced.objective.di),
            "art": float(best_result.best_balanced.objective.art),
            "max_rt_p99": float(best_result.best_balanced.objective.art),
            "score_balanced": float(opt_score),
            "pareto_row": asdict(selected_row) if selected_row is not None else None,
        },
    }

    payload = {
        "scenario": scenario_name,
        "input_csv": str(csv_path),
        "training_backend": current_training_backend(),
        "gpu_available": GPU_AVAILABLE,
        "selected_run": best_run_meta,
        "runs": summaries,
        "base_objective": {"source": "historical_csv_ground_truth", "di": base_obj.di, "art": base_obj.art, "max_rt_p99": base_obj.art, "score_balanced": base_score},
        "optimized_objective": {"source": "simulated_routing_replay", "di": opt_obj.di, "art": opt_obj.art, "max_rt_p99": opt_obj.art, "score_balanced": opt_score},
        "validation_steps": validation_steps,
        "result": {
            "generated_at": best_result.generated_at,
            "sample_count": best_result.sample_count,
            "used_samples": best_result.used_samples,
            "config": asdict(best_result.config),
            "best_balanced": {
                "params": best_result.best_balanced.params,
                "objective": {
                    "di": best_result.best_balanced.objective.di,
                    "art": best_result.best_balanced.objective.art,
                    "max_rt_p99": best_result.best_balanced.objective.art,
                    "score_balanced": opt_score,
                },
            },
            "best_di": {
                "params": best_result.best_di.params,
                "objective": {
                    "di": best_result.best_di.objective.di,
                    "art": best_result.best_di.objective.art,
                    "max_rt_p99": best_result.best_di.objective.art,
                },
            },
            "best_art": {
                "params": best_result.best_art.params,
                "objective": {
                    "di": best_result.best_art.objective.di,
                    "art": best_result.best_art.objective.art,
                    "max_rt_p99": best_result.best_art.objective.art,
                },
            },
            "best_max_rt": {
                "params": best_result.best_art.params,
                "objective": {
                    "di": best_result.best_art.objective.di,
                    "art": best_result.best_art.objective.art,
                    "max_rt_p99": best_result.best_art.objective.art,
                },
            },
            "archive": archive_export,
            "archive_trace": [asdict(row) for row in best_result.archive_trace],
            "convergence_plot": str(convergence_out),
            "pareto_plot": str(pareto_out),
            "normalization_plot": str(normalized_out),
            "comparison_plot": str(comparison_out),
            "score_history": best_result.score_history,
            "iter_best_history": best_result.iter_best_history,
            "iter_mean_history": best_result.iter_mean_history,
            "archive_size_history": best_result.archive_size_history,
            "stagnation_events": best_result.stagnation_events,
        },
    }

    with report_out.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return {
        "scenario": scenario_name,
        "params_out": str(params_out),
        "report_out": str(report_out),
        "sample_count": ds.sample_count,
        "used_samples": ds.used_samples,
        "training_backend": current_training_backend(),
        "gpu_available": GPU_AVAILABLE,
        "base_score": base_score,
        "opt_score": opt_score,
        "convergence_plot": str(convergence_out),
        "pareto_plot": str(pareto_out),
        "normalization_plot": str(normalized_out),
        "comparison_plot": str(comparison_out),
    }


def run_batch_scenarios(
    csv_files: Sequence[str | Path],
    base_params_path: Optional[str | Path] = None,
    out_dir: str | Path = "./outputs",
    particles: int = 30,
    iterations: int = 3000,
    spread: float = 8.0,
    seed: int = 0,
    runs: int = 5,
    allow_regression: bool = False,
) -> List[Dict]:
    base_params = load_base_params(base_params_path)
    base_params = repair_params(base_params)

    outputs: List[Dict] = []
    scenario_progress = tqdm(csv_files, desc="Scenarios", unit="csv", leave=False)
    for csv_file in scenario_progress:
        summary = run_scenario_optimization(
            csv_path=csv_file,
            base_params=base_params,
            out_dir=out_dir,
            particles=particles,
            iterations=iterations,
            spread=spread,
            seed=seed,
            runs=runs,
            allow_regression=allow_regression,
        )
        outputs.append(summary)
    return outputs


## Setup Skenario
Isi `CSV_FILES` dan `BASE_PARAMS_JSON` sesuai file yang ada di Colab.


In [ ]:
from IPython.display import display, Markdown, Image
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Google Drive mount dilewati: {exc}")

path_skripsi = '/content/drive/MyDrive/skripsi'
if os.path.exists(path_skripsi):
    os.chdir(path_skripsi)
    print(f"Berhasil terhubung. Direktori kerja sekarang: {os.getcwd()}")
else:
    print(f"Folder '{path_skripsi}' tidak ditemukan. Pastikan nama folder sudah benar.")

backend_name = set_training_backend(True)
print(f"Training backend aktif: {backend_name} | GPU tersedia: {GPU_AVAILABLE}")
if not GPU_AVAILABLE:
    print("GPU belum terdeteksi. Pastikan Colab runtime menggunakan GPU, lalu rerun notebook dari awal.")

CSV_FILES = [
    "normal5.csv",
]
BASE_PARAMS_JSON = None  # contoh: "/content/base_fuzzy_params.json"
OUT_DIR = "./mopso_outputs"

if not CSV_FILES:
    print("Isi CSV_FILES terlebih dahulu, lalu jalankan ulang cell ini.")


## Step 1 - Data Loading & Ground Truth

Load CSV, tampilkan tabel quality report, lalu plot gambaran awal dataset.


In [ ]:
scenario_csv = CSV_FILES[0]
ds = load_offline_samples_csv(scenario_csv)
base_params = repair_params(load_base_params(BASE_PARAMS_JSON))
quality = build_dataset_quality_report(ds)
overview_plot = plot_dataset_overview(ds, ds.scenario_name, OUT_DIR)

quality_rows = []
for metric, value in quality.items():
    if isinstance(value, dict):
        quality_rows.append({"metric": metric, **value})
    else:
        quality_rows.append({"metric": metric, "value": value})
quality_df = pd.DataFrame(quality_rows)

display(Markdown(f"### Dataset Loaded: `{scenario_csv}`"))
display(quality_df)
display(Image(filename=overview_plot))


## Step 2 - Kondisi Awal (Sebelum Optimasi)

Tampilkan membership function base dan snapshot trace replay untuk melihat perilaku sistem sebelum training.


In [ ]:
system_constants = extract_system_constants(ds)
base_trace = build_objective_validation_trace(base_params, ds, max_rows=8, constants=system_constants)
base_hist_obj = evaluate_historical_base_objective(ds)
base_sim_obj = evaluate_dataset_di_art(base_params, ds, system_constants)

base_membership_plot = plot_fuzzy_membership(
    base_params,
    title=f"{ds.scenario_name} - Base Membership",
    out_dir=OUT_DIR,
)
base_snapshot_plot = plot_snapshot_trace(
    base_params,
    ds,
    system_constants,
    title=f"{ds.scenario_name} - Base Snapshot Trace",
    out_dir=OUT_DIR,
)

display(Markdown("### Membership Function Awal"))
display(Image(filename=base_membership_plot))

display(Markdown("### Snapshot Trace Awal"))
display(Image(filename=base_snapshot_plot))

display(Markdown("### Ringkasan Objective Base"))
display(pd.DataFrame([
    {"source": "Historical CSV Ground Truth", "di_ratio": base_hist_obj.di, "art_mean_rt_ms": base_hist_obj.art},
    {"source": "Simulated Routing Replay", "di_ratio": base_sim_obj.di, "art_mean_rt_ms": base_sim_obj.art},
]).round(6))

display(Markdown("### Konstanta Kalibrasi Sistem"))
display(pd.DataFrame([base_trace["constants"]]).round(6))


## Step 3 - Proses Training MOPSO

Progress bar training berjalan di cell ini.


In [ ]:
result_summary = run_scenario_optimization(
    csv_path=scenario_csv,
    base_params=base_params,
    out_dir=OUT_DIR,
    particles=NUM_PARTICLES_DEFAULT,
    iterations=ITERATIONS_DEFAULT,
    spread=8.0,
    seed=0,
    runs=5,
    allow_regression=False,
)

display(pd.DataFrame([
    {
        "scenario": result_summary["scenario"],
        "used_samples": result_summary["used_samples"],
        "training_backend": result_summary["training_backend"],
        "gpu_available": result_summary["gpu_available"],
        "base_score": result_summary["base_score"],
        "opt_score": result_summary["opt_score"],
    }
]).round(6))
print("Training selesai. Report:", result_summary["report_out"])
print("Optimized params:", result_summary["params_out"])


## Step 4 - Evaluasi Pareto & Konvergensi

Tampilkan kurva konvergensi, sebaran Pareto, dan normalisasi archive.


In [ ]:
with open(result_summary["report_out"], "r", encoding="utf-8") as f:
    report = json.load(f)

display(Markdown("### Ringkasan Run Terpilih"))
display(pd.DataFrame([result_summary]).round(6))

display(Markdown("### Top 10 Archive Candidates"))
display(pd.DataFrame(report["result"]["archive_trace"][:10]).rename(columns={
    "di": "di_ratio",
    "norm_di": "norm_di_ratio",
    "score": "selection_score",
}).round(6))

display(Image(filename=report["result"]["convergence_plot"]))
display(Image(filename=report["result"]["pareto_plot"]))
display(Image(filename=report["result"]["normalization_plot"]))


## Step 5 - Kondisi Akhir (Setelah Optimasi)

Tampilkan membership function hasil JSON optimized, snapshot trace optimized, lalu bandingkan base vs optimized.


In [ ]:
optimized_params = repair_params(load_base_params(result_summary["params_out"]))
optimized_membership_plot = plot_fuzzy_membership(
    optimized_params,
    title=f"{ds.scenario_name} - Optimized Membership",
    out_dir=OUT_DIR,
)
optimized_snapshot_plot = plot_snapshot_trace(
    optimized_params,
    ds,
    system_constants,
    title=f"{ds.scenario_name} - Optimized Snapshot Trace",
    out_dir=OUT_DIR,
)

display(Markdown("### Membership Function Optimized"))
display(Image(filename=optimized_membership_plot))

display(Markdown("### Snapshot Trace Optimized"))
display(Image(filename=optimized_snapshot_plot))

display(Markdown("### Base vs Optimized"))
display(pd.DataFrame([
    {"metric": "DI Ratio (Capacity-Normalized CPU)", "base": report["base_objective"]["di"], "optimized": report["optimized_objective"]["di"]},
    {"metric": "ART Mean RT (ms)", "base": report["base_objective"]["art"], "optimized": report["optimized_objective"]["art"]},
    {"metric": "Selection Score (DI-first)", "base": report["base_objective"]["score_balanced"], "optimized": report["optimized_objective"]["score_balanced"]},
]).round(6))
display(Image(filename=report["result"]["comparison_plot"]))

display(Markdown("### Solusi Terpilih"))
display(pd.DataFrame([report["result"]["best_balanced"]["objective"]]).rename(columns={"di": "di_ratio", "score_balanced": "selection_score"}).round(6))
display(pd.DataFrame([report["validation_steps"]["step_6_selected_solution"]]).rename(columns={"di": "di_ratio", "score_balanced": "selection_score"}))
print("Report tersimpan di:", result_summary["report_out"])
print("Params tersimpan di:", result_summary["params_out"])
